# **HuggingFace Login**

In [ ]:
from huggingface_hub import login

login()  # You will be prompted for your HF key, which will then be saved locally

# **Installs**

In [ ]:
!pip install langchain-huggingface langchain-community huggingface_hub transformers datasets langchain-yt-dlp youtube-transcript-api faiss-cpu

# **Imports**

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEndpointEmbeddings
from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# **Step 1**

# **Indexing**

In [ ]:
video_id='1a1VXDdIyrk'

loader = YoutubeLoader(
    video_id=video_id
)

document = loader.load()

In [ ]:
document

# **Text Splitter**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50)
chunks = text_splitter.create_documents([document[0].page_content])

In [ ]:
document = loader.load()

document

In [ ]:
chunks = text_splitter.create_documents([document[0].page_content])

In [ ]:
chunks

# **Embed Chunks | Store in VectorDB (FAISS)**

In [ ]:
embed_model = HuggingFaceEndpointEmbeddings(
    repo_id = 'ibm-granite/granite-embedding-311m-multilingual-r2',
    task = 'feature-extraction'
)

In [ ]:
vector_store = FAISS.from_documents(chunks, embed_model)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['8642e907-fa16-4252-b5dd-88c70d9e34e6'])

# **Step 2**

# **Retriever**

In [ ]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k':4})

In [ ]:
retriever

In [ ]:
retriever.invoke('What is Harness')

# **Prompt Augmentation**

In [ ]:
prompt = PromptTemplate(
    template = """
    ""
      You are a very helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say in a polite manner that you don't know about it. Don't Make up or Hallucinate Any Answer from yourself.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "What's harness engineering and how is it helpful and what's next after harness engineering?"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text = " \n ".join(rd.page_content for rd in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({'context':context_text, 'question':question})

# **Model Download**

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-huggingface \
transformers \
accelerate \
torch \
faiss-cpu \
youtube-transcript-api \
langchain-yt-dlp

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
from transformers import pipeline

pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    return_full_text=False,
    temperature=0.3,
    do_sample=True
)

In [ ]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(
    pipeline=pipe
)

# **Generation**

In [ ]:
# model = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.3",
#     task="conversational",
#     max_new_tokens=512,
#     do_sample=False,
#     repetition_penalty=1.03,
# )
# Because the inference of these models weren't available, so a model is downloaded using Torch Pipeline

chat = ChatHuggingFace(llm=llm, verbose=True)

In [ ]:
# This code is just to check the available inference servers on huggingface to run a model


# from huggingface_hub import model_info
# # Change this to the model you want to test
# info = model_info("mistralai/Mistral-7B-Instruct-v0.3", expand="inference") #
# print(info.transformers_info)


In [ ]:
final_prompt

In [ ]:
output = chat.invoke(final_prompt)

In [ ]:
output

In [ ]:
final_output = output.content

In [ ]:
final_output

# **Building Every Component in a Chain with Lesser Code**

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers.string import StrOutputParser

write a function to format the retriever output (list of documents)

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [65]:
parallel_chain.invoke('What is Harness Engineering?')

{'context': "doesn't really help us understand what it is and what it isn't. How exactly is harness engineering\n\nunderneath. One clarification to be made here is that harness engineering doesn't necessarily\n\nMore on them later. To put it simply, harness engineering actually existed before the term harness\n\nis that harness engineering doesn't necessarily deprecate context engineering and it certainly",
 'question': 'What is Harness Engineering?'}

In [68]:
parser = StrOutputParser()

In [69]:
chain = parallel_chain | prompt | chat | parser

In [70]:
chain.invoke('Please summarize the video')

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'The video discusses advancements in the AI industry and mentions a need to work quickly and efficiently. It also references "cursor," suggesting there might be ongoing projects involving this entity, but does not provide specific details about their work. The speaker encourages viewers to keep an eye on "cursor" for more information in the future.'

**Finished, Thank You**

**Don't Forget to follow on:**

**> linkedin.com/in/hafizmusanadeem**

**> github.com/hafizmusanadeem**